# 04 — Frozen test retrieval predictions

Run only after the protocol and validation selection are locked. This notebook loads no annotation contents and generates rankings over identical candidate pools. Training references are analogies, not claims about the query patient. Unknown test concepts do not expand the training vocabulary.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
assert (ROOT / 'core.py').exists(), 'Start the notebook kernel in graph_reasoning/'
sys.path.insert(0, str(ROOT))
from core import *
CONFIG = load(ROOT / 'config.json')
OUT = ROOT / CONFIG['output_dir']
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import torch
from learning import Ranker, rank
frozen=load(OUT/'freeze.json')
for f,h in frozen['artifacts'].items(): assert digest(OUT/f)==h, f'Changed artifact: {f}'
for f,h in frozen['source'].items(): assert digest(ROOT/f)==h, f'Changed source: {f}'
rows=load(OUT/'records.json'); graph=load(OUT/'graph.json'); pools=load(OUT/'pools.json')
predictions=[]
for seed in frozen['seeds']:
    models={}
    for arm,kind,file in [('entity_mlp','mlp',f'mlp_{seed}.pt'),('learned_graph',frozen['selected'],f"{frozen['selected']}_{seed}.pt"),('shuffled_graph',frozen['selected'],f'shuffled_{seed}.pt')]:
        model=Ranker(len(graph['vocabulary']),kind)
        model.load_state_dict(torch.load(OUT/file,weights_only=True)); model.eval(); models[arm]=model
    for q in [r for r in rows if r['split']=='test']:
        arms={'similarity':rank(rows,graph,q),'rule_graph':rank(rows,graph,q,rule=True)}
        arms.update({arm:rank(rows,graph,q,model=model,shuffle=arm=='shuffled_graph',seed=seed) for arm,model in models.items()})
        for arm,ranking in arms.items():
            assert set(ranking)==set(pools[q['id']]), 'Candidate pool changed'
            predictions.append(dict(query_id=q['id'],patient_id=q['patient_id'],modality=q['modality'],seed=seed,arm=arm,ranking=ranking))
save(OUT/'rankings.json',predictions)
print('Saved rankings:',len(predictions),'; synthetic:',frozen['synthetic'])
